In [1]:
import joblib

X_train = joblib.load(
    "X_train_strat.pkl"
)

X_test = joblib.load(
    "X_test_strat.pkl"
)

y_train = joblib.load(
    "y_train_strat.pkl"
)

y_test = joblib.load(
    "y_test_strat.pkl"
)

In [2]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import RobustScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_regression
)

In [3]:
preprocessor_robust = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="mean")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=mutual_info_regression,
            k=1000
        )
    )

])

In [4]:
from lightgbm import LGBMRegressor

from xgboost import XGBRegressor

from catboost import CatBoostRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.neural_network import MLPRegressor

In [6]:
# MODEL PIPELINES


lgbm_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        LGBMRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


xgb_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        XGBRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


cat_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        CatBoostRegressor(
            iterations=300,
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )
    )

])


rf_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            max_depth=6
        )
    )

])


svr_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1
        )
    )

])


mlp_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        MLPRegressor(

            hidden_layer_sizes=(256,128),

            activation='relu',

            solver='adam',

            alpha=0.0001,

            batch_size='auto',

            learning_rate='adaptive',

            learning_rate_init=0.0001,

            max_iter=500,

            early_stopping=True,

            random_state=42

        )
    )

])


et_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        ExtraTreesRegressor(

            n_estimators=300,

            random_state=42,

            n_jobs=-1,

            max_depth=10

        )
    )

])


hist_pipeline = Pipeline([

    ("preprocessing", preprocessor_robust),

    (
        "model",
        HistGradientBoostingRegressor(

            max_iter=300,

            learning_rate=0.05,

            max_depth=6,

            random_state=42

        )
    )

])

models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,

    "ExtraTrees": et_pipeline,

    "HistGradient": hist_pipeline,

    "SVR": svr_pipeline,

    "MLPRegressor": mlp_pipeline

}

In [7]:
from sklearn.model_selection import cross_val_score

In [8]:
from sklearn.model_selection import cross_val_score

results = []

for model_name, pipeline in models.items():

    cv_scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    results.append({

        "Model": model_name,

        "CV Mean R2": cv_scores.mean(),

        "CV Std": cv_scores.std()

    })

   
    print(model_name)
   

    print("CV Scores:")
    print(cv_scores)

    print("\nMean CV R2:")
    print(cv_scores.mean())

LightGBM
CV Scores:
[0.80064886 0.77196584 0.80937763 0.78416732 0.76432582]

Mean CV R2:
0.7860970933837359
XGBoost
CV Scores:
[0.77070929 0.74900589 0.78178593 0.7599725  0.73941536]

Mean CV R2:
0.7601777938956126
CatBoost
CV Scores:
[0.79244919 0.76616212 0.79953997 0.78189336 0.76730862]

Mean CV R2:
0.7814706496055982
RandomForest
CV Scores:
[0.73132071 0.72839977 0.73993874 0.74444093 0.71317315]

Mean CV R2:
0.7314546621918634
ExtraTrees
CV Scores:
[0.76994853 0.75003626 0.77378481 0.77361001 0.75127833]

Mean CV R2:
0.7637315873058096
HistGradient
CV Scores:
[0.80494458 0.77918987 0.80530251 0.79195459 0.77213402]

Mean CV R2:
0.790705113835086
SVR
CV Scores:
[0.46585186 0.40493236 0.47040814 0.39208588 0.45061571]

Mean CV R2:
0.4367787900539966
MLPRegressor
CV Scores:
[ 0.57658044  0.23623512  0.65185557  0.48987978 -1.07850906]

Mean CV R2:
0.17520837178667312


In [9]:
import pandas as pd

In [10]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="CV Mean R2",

    ascending=False

)

results_df

,Model,CV Mean R2,CV Std
5,HistGradient,0.790705,0.013378
0,LightGBM,0.786097,0.016918
2,CatBoost,0.781471,0.013282
4,ExtraTrees,0.763732,0.010770
1,XGBoost,0.760178,0.015058
3,RandomForest,0.731455,0.010809
6,SVR,0.436779,0.032185
7,MLPRegressor,0.175208,0.642315


In [11]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

In [12]:
import numpy as np

In [13]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

test_results = []

for model_name, pipeline in models.items():

    # TRAIN MODEL
  

    pipeline.fit(
        X_train,
        y_train
    )



    
    # TRAIN PREDICTIONS
    

    y_train_pred = pipeline.predict(
        X_train
    )



    
    # TEST PREDICTIONS
    

    y_test_pred = pipeline.predict(
        X_test
    )



    
    # TRAIN METRICS
    

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred
        )
    )



    
    # TEST METRICS
    

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )



    
    # STORE RESULTS
    

    test_results.append({

        "Model": model_name,

        "Train_R2": train_r2,

        "Test_R2": test_r2,

        "Train_RMSE": train_rmse,

        "Test_RMSE": test_rmse,

        "Test_MAE": test_mae

    })



    
    # PRINT RESULTS
    

    print("\n")
    print(model_name)
    

    print("Train R2 :", train_r2)
    print("Test R2  :", test_r2)

    print("Train RMSE :", train_rmse)
    print("Test RMSE  :", test_rmse)

    print("Test MAE :", test_mae)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.069780 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 242915
[LightGBM] [Info] Number of data points in the train set: 7984, number of used features: 1000
[LightGBM] [Info] Start training from score -2.891766


c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(




LightGBM
Train R2 : 0.9567940114343697
Test R2  : 0.7971981508022652
Train RMSE : 0.49173191805132815
Test RMSE  : 1.0703434478810034
Test MAE : 0.6980140995857047


XGBoost
Train R2 : 0.971280026693173
Test R2  : 0.7805317603914881
Train RMSE : 0.400911433343727
Test RMSE  : 1.113455947993983
Test MAE : 0.728804480566969


CatBoost
Train R2 : 0.9142814563334709
Test R2  : 0.7977577136887481
Train RMSE : 0.6926180698975896
Test RMSE  : 1.0688658031387603
Test MAE : 0.7177752181139954


RandomForest
Train R2 : 0.7797044990100236
Test R2  : 0.7310610046851873
Train RMSE : 1.1103483198620772
Test RMSE  : 1.232577082582853
Test MAE : 0.8811433363543698


ExtraTrees
Train R2 : 0.888951990425826
Test R2  : 0.7665558324673326
Train RMSE : 0.7883368272690174
Test RMSE  : 1.1483617367356358
Test MAE : 0.7894820845568624


HistGradient
Train R2 : 0.9189986358891498
Test R2  : 0.801238236253315
Train RMSE : 0.6732906635540714
Test RMSE  : 1.059628475121704
Test MAE : 0.7048039359020434


SVR
Tr

In [14]:
results_df = pd.DataFrame(
    test_results
)

results_df = results_df.sort_values(

    by="Test_R2",

    ascending=False

)

results_df

,Model,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Test_MAE
5,HistGradient,0.918999,0.801238,0.673291,1.059628,0.704804
2,CatBoost,0.914281,0.797758,0.692618,1.068866,0.717775
0,LightGBM,0.956794,0.797198,0.491732,1.070343,0.698014
1,XGBoost,0.971280,0.780532,0.400911,1.113456,0.728804
4,ExtraTrees,0.888952,0.766556,0.788337,1.148362,0.789482
3,RandomForest,0.779704,0.731061,1.110348,1.232577,0.881143
6,SVR,0.478728,0.472702,1.708003,1.725898,1.299786
7,MLPRegressor,0.703844,-82831.795780,1.287410,684.050807,16.244059
